# WasteLens - Iteration 12 (Colab GPU)


Protocol: docs/rejection_experiment/iteration12_distillation_protocol.md (registered pre-training, commit 3b88e70). Run every code cell top to bottom; every heavy cell STOPS the run on any non-zero exit - nothing may be skipped, reordered, or re-run after completion.


## S0 - environment


In [ ]:
!nvidia-smi


In [ ]:
import sys, platform
import tensorflow as tf
print('Python:', sys.version.split()[0], '|', platform.platform())
print('TensorFlow:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPUs:', gpus if gpus else 'NONE - enable a GPU runtime')
assert gpus, 'STOP: no GPU visible'
import numpy, sklearn, PIL
print('numpy:', numpy.__version__,
      '| sklearn:', sklearn.__version__,
      '| pillow:', PIL.__version__)


Python: 3.13.15 | Linux-6.6.122+-x86_64-with-glibc2.39
TensorFlow: 2.20.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
numpy: 2.1.3 | sklearn: 1.6.1 | pillow: 11.3.0


## S1 - pin the repo: checkout the registered commit


In [ ]:
import os, subprocess, sys
from pathlib import Path

PINNED = 'cf2787b0746506256771a54c3c69feaf6ce3618e'
REPO = '/content/WasteLens'

if not Path(REPO, 'src', 'train_distill_adaptation.py').exists():
    subprocess.run(['git', 'clone',
                    'https://github.com/Anik00223/WasteLens.git',
                    REPO], check=True)
os.chdir(REPO)
subprocess.run(['git', 'fetch', 'origin'], check=True)
subprocess.run(['git', 'checkout', '--force', PINNED], check=True)

head = subprocess.run(['git', 'rev-parse', 'HEAD'],
                      capture_output=True, text=True,
                      check=True).stdout.strip()
origin = subprocess.run(['git', 'rev-parse', 'origin/main'],
                        capture_output=True, text=True,
                        check=True).stdout.strip()
print('HEAD        :', head)
print('PINNED      :', PINNED)
print('origin/main :', origin, '(informational - the pin wins)')
assert head == PINNED, ('STOP: HEAD ' + head + ' != PINNED '
                        + PINNED)
MANIFESTS = ['docs/rejection_experiment/adaptation_set_manifest.json',
             'docs/rejection_experiment/fresh_set_manifest.json']
print('repo pinned OK')


HEAD        : cf2787b0746506256771a54c3c69feaf6ce3618e
PINNED      : cf2787b0746506256771a54c3c69feaf6ce3618e
origin/main : 63c67d3dd653b93e0c10a502e13d7ce6eddf6891 (informational - the pin wins)
repo pinned OK


In [ ]:
for rel in MANIFESTS:
    attrs = subprocess.run(['git', 'check-attr', 'text', 'eol', '--', rel],
                           capture_output=True, text=True,
                           check=True).stdout
    print(attrs.strip())
    assert 'eol: crlf' in attrs, (
        'STOP: ' + rel + ' is not checked out with the registered CRLF bytes (the dataset locks are MD5s of the CRLF form)')
print('manifest EOL pin OK')

def run(cmd, log=None):
    """Run a repo command; any non-zero exit stops the cell."""
    cmd = [str(c) for c in cmd]
    print('+', ' '.join(cmd), flush=True)
    if log:
        with open(log, 'a', encoding='utf-8') as fh:
            p = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE,
                                 stderr=subprocess.STDOUT, text=True, bufsize=1)
            for line in p.stdout:
                print(line, end='', flush=True)
                fh.write(line)
            rc = p.wait()
        if rc:
            raise SystemExit('STOP rc=' + str(rc) + ': ' + ' '.join(cmd) + ' (log: ' + log + ')')
    else:
        rc = subprocess.run(cmd, cwd=REPO).returncode
        if rc:
            raise SystemExit('STOP rc=' + str(rc) + ': ' + ' '.join(cmd))

PY = sys.executable
print('runner ready; PY =', PY)


docs/rejection_experiment/adaptation_set_manifest.json: text: set
docs/rejection_experiment/adaptation_set_manifest.json: eol: crlf
docs/rejection_experiment/fresh_set_manifest.json: text: set
docs/rejection_experiment/fresh_set_manifest.json: eol: crlf
manifest EOL pin OK
runner ready; PY = /usr/bin/python3


## S2 - Drive mount + teacher restore (md5-gated)

One-time manual step: the shipped checkpoint is gitignored, so a fresh runtime does not have it. Upload
`models/checkpoints/wastelens_rej_shipped_best.keras` (md5 `8735019226412c4d67b0a17269dc38ad`, 13,583,639 B) from the laptop repo to `MyDrive/WasteLens/iteration12/checkpoints/` before S5/S6. The cell fails loudly with the exact instructions if it is absent - training never starts from another model.


In [6]:
from google.colab import drive
drive.mount('/content/drive')
BK = '/content/drive/MyDrive/WasteLens/iteration12'
for sub in ('datasets/adaptation_set', 'datasets/fresh_set',
            'checkpoints', 'evidence', 'logs'):
    os.makedirs(BK + '/' + sub, exist_ok=True)
print('backup root:', BK)
run([PY, 'src/restore_teacher_ckpt.py', '--drive', BK])


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


NameError: name 'os' is not defined

In [3]:
from google.colab import drive
from pathlib import Path
import os
import subprocess
import sys

drive.mount("/content/drive")

BK = "/content/drive/MyDrive/WasteLens/iteration12"

for sub in (
    "datasets/adaptation_set",
    "datasets/fresh_set",
    "checkpoints",
    "evidence",
    "logs",
):
    os.makedirs(os.path.join(BK, sub), exist_ok=True)

print("backup root:", BK)

result = subprocess.run(
    [
        sys.executable,
        "src/restore_teacher_ckpt.py",
        "--drive",
        BK,
    ],
    cwd="/content/WasteLens",
    text=True,
    capture_output=True,
)

print("===== STDOUT =====")
print(result.stdout)

print("===== STDERR =====")
print(result.stderr)

print("===== RETURN CODE =====")
print(result.returncode)

if result.returncode != 0:
    raise RuntimeError("Teacher restore failed.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
backup root: /content/drive/MyDrive/WasteLens/iteration12


FileNotFoundError: [Errno 2] No such file or directory: '/content/WasteLens'

In [4]:
from pathlib import Path
import hashlib

p = Path(
    "/content/WasteLens/models/checkpoints/"
    "wastelens_rej_shipped_best.keras"
)

print("Exists:", p.exists())

if p.exists():
    h = hashlib.md5()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)

    print("Size:", p.stat().st_size)
    print("MD5:", h.hexdigest())
    print(
        "MD5 PASS:",
        h.hexdigest() == "8735019226412c4d67b0a17269dc38ad"
    )

Exists: False


In [5]:
from pathlib import Path
import subprocess

REPO = Path("/content/WasteLens")
PINNED = "cf2787b0746506256771a54c3c69feaf6ce3618e"

if not REPO.exists():
    print("Cloning WasteLens...")
    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/Anik00223/WasteLens.git",
            str(REPO),
        ],
        check=True,
    )
else:
    print("WasteLens directory already exists.")

subprocess.run(
    ["git", "fetch", "origin", "main"],
    cwd=REPO,
    check=True,
)

subprocess.run(
    ["git", "checkout", "--force", PINNED],
    cwd=REPO,
    check=True,
)

head = subprocess.run(
    ["git", "rev-parse", "HEAD"],
    cwd=REPO,
    capture_output=True,
    text=True,
    check=True,
).stdout.strip()

print("HEAD:", head)
print("PINNED:", PINNED)

assert head == PINNED, "STOP: wrong repository commit"

print("✅ Repository restored and pinned")

Cloning WasteLens...
HEAD: cf2787b0746506256771a54c3c69feaf6ce3618e
PINNED: cf2787b0746506256771a54c3c69feaf6ce3618e
✅ Repository restored and pinned


In [6]:
from pathlib import Path

repo = Path("/content/WasteLens")

for p in [
    repo / "src/restore_teacher_ckpt.py",
    repo / "src/materialize_manifest.py",
    repo / "src/iter12_preflight.py",
    repo / "src/train_distill_adaptation.py",
]:
    print(
        p,
        "→",
        "FOUND ✅" if p.exists() else "MISSING ❌"
    )

/content/WasteLens/src/restore_teacher_ckpt.py → FOUND ✅
/content/WasteLens/src/materialize_manifest.py → FOUND ✅
/content/WasteLens/src/iter12_preflight.py → FOUND ✅
/content/WasteLens/src/train_distill_adaptation.py → FOUND ✅


In [7]:
from pathlib import Path
import subprocess
import sys

REPO = Path("/content/WasteLens")
BK = "/content/drive/MyDrive/WasteLens/iteration12"

result = subprocess.run(
    [
        sys.executable,
        "src/restore_teacher_ckpt.py",
        "--drive",
        BK,
    ],
    cwd=REPO,
    text=True,
    capture_output=True,
)

print("===== STDOUT =====")
print(result.stdout)

print("===== STDERR =====")
print(result.stderr)

print("===== RETURN CODE =====")
print(result.returncode)

if result.returncode != 0:
    raise RuntimeError("Teacher restore failed.")

print("✅ Teacher restore completed")

===== STDOUT =====
teacher target: models/checkpoints/wastelens_rej_shipped_best.keras
  local md5   : MISSING  (want 8735019226412c4d67b0a17269dc38ad)
  drive md5   : 8735019226412c4d67b0a17269dc38ad  (/content/drive/MyDrive/WasteLens/iteration12/checkpoints/wastelens_rej_shipped_best.keras)
teacher: restored /content/drive/MyDrive/WasteLens/iteration12/checkpoints/wastelens_rej_shipped_best.keras -> models/checkpoints/wastelens_rej_shipped_best.keras (md5 verified)

===== STDERR =====

===== RETURN CODE =====
0
✅ Teacher restore completed


In [8]:
from pathlib import Path
import hashlib

p = Path(
    "/content/WasteLens/models/checkpoints/"
    "wastelens_rej_shipped_best.keras"
)

print("Exists:", p.exists())

if p.exists():
    h = hashlib.md5()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)

    actual = h.hexdigest()

    print("Size:", p.stat().st_size, "bytes")
    print("MD5:", actual)
    print(
        "MD5 PASS:",
        actual == "8735019226412c4d67b0a17269dc38ad"
    )

Exists: True
Size: 13583639 bytes
MD5: 8735019226412c4d67b0a17269dc38ad
MD5 PASS: True


In [9]:
from pathlib import Path
import subprocess
import sys

REPO = Path("/content/WasteLens")
BK = "/content/drive/MyDrive/WasteLens/iteration12"

commands = [
    [
        sys.executable,
        "src/materialize_manifest.py",
        "--manifest",
        "docs/rejection_experiment/adaptation_set_manifest.json",
        "--out",
        "scratch/adaptation_set",
        "--drive",
        BK + "/datasets/adaptation_set",
        "--report",
        BK + "/datasets/adaptation_lock_report.json",
    ],
    [
        sys.executable,
        "src/materialize_manifest.py",
        "--manifest",
        "docs/rejection_experiment/fresh_set_manifest.json",
        "--out",
        "scratch/fresh_set",
        "--drive",
        BK + "/datasets/fresh_set",
        "--report",
        BK + "/datasets/fresh_lock_report.json",
    ],
]

for cmd in commands:
    print("\nRunning:")
    print(" ".join(cmd))

    result = subprocess.run(
        cmd,
        cwd=REPO,
        text=True,
        capture_output=True,
    )

    print(result.stdout)
    print(result.stderr)

    if result.returncode != 0:
        raise RuntimeError(
            f"Dataset materialization failed with rc={result.returncode}"
        )

print("\n✅ BOTH DATASETS MATERIALIZED SUCCESSFULLY")


Running:
/usr/bin/python3 src/materialize_manifest.py --manifest docs/rejection_experiment/adaptation_set_manifest.json --out scratch/adaptation_set --drive /content/drive/MyDrive/WasteLens/iteration12/datasets/adaptation_set --report /content/drive/MyDrive/WasteLens/iteration12/datasets/adaptation_lock_report.json
materialize adaptation_set_manifest.json: 94/94 verified, 0 missing, 0 mismatched; report -> /content/drive/MyDrive/WasteLens/iteration12/datasets/adaptation_lock_report.json



Running:
/usr/bin/python3 src/materialize_manifest.py --manifest docs/rejection_experiment/fresh_set_manifest.json --out scratch/fresh_set --drive /content/drive/MyDrive/WasteLens/iteration12/datasets/fresh_set --report /content/drive/MyDrive/WasteLens/iteration12/datasets/fresh_lock_report.json
materialize fresh_set_manifest.json: 120/131 verified, 0 missing, 11 mismatched; report -> /content/drive/MyDrive/WasteLens/iteration12/datasets/fresh_lock_report.json

INCOMPLETE: 0 missing, 11 mismatched -

RuntimeError: Dataset materialization failed with rc=1

In [11]:
import subprocess
import sys

result = subprocess.run(
    [
        sys.executable,
        "src/iter12_preflight.py",
    ],
    cwd="/content/WasteLens",
    text=True,
    capture_output=True,
)

print(result.stdout)
print(result.stderr)
print("RETURN CODE:", result.returncode)

if result.returncode != 0:
    raise RuntimeError("❌ Iteration-12 preflight FAILED")

print("✅ ITERATION-12 PREFLIGHT PASSED")

PASS adapt_manifest_md5: got acae0933aca123c835c1e55396af182a want acae0933aca123c835c1e55396af182a
PASS adapt_manifest_entries: got 94 want 94
PASS adapt_dir_file_count: got 94 want 94
PASS adapt_files_byte_locked: got {'verified': 94, 'bad_total': 0} want {'verified': 94}
PASS fresh_manifest_md5: got 99540f56efb95e2c26b2b6cc9414ff5d want 99540f56efb95e2c26b2b6cc9414ff5d
PASS fresh_manifest_entries: got 131 want 131
FAIL fresh_dir_file_count: got 120 want 131
FAIL fresh_files_byte_locked: got {'verified': 120, 'bad_total': 11} want {'verified': 131}
PASS teacher_md5: got 8735019226412c4d67b0a17269dc38ad want 8735019226412c4d67b0a17269dc38ad
preflight FAILED: 2 check(s) - ['fresh_dir_file_count', 'fresh_files_byte_locked']


RETURN CODE: 1


RuntimeError: ❌ Iteration-12 preflight FAILED

In [12]:
from pathlib import Path

d = Path(
    "/content/drive/MyDrive/WasteLens/iteration12/datasets/fresh_set"
)

names = [
    "supported-organic__Compost__05.jpg",
    "supported-organic__Compost_bins__02.jpg",
    "supported-recyclable__Aluminium_cans__01.jpg",
    "supported-hazardous__Button_cells__00.jpg",
    "supported-hazardous__Button_cells__03.jpg",
    "supported-hazardous__Fluorescent_lamps__01.jpg",
    "supported-hazardous__Fluorescent_lamps__02.jpg",
    "ambiguous__Pizza_boxes__00.jpg",
    "ood-objects__Toy_robots__02.png",
    "ood-clothes__Shirts__07.jpg",
    "multi-object__Beach_litter__03.jpg",
]

for name in names:
    print(
        name,
        "→ FOUND ✅" if (d / name).exists() else "MISSING ❌"
    )

supported-organic__Compost__05.jpg → FOUND ✅
supported-organic__Compost_bins__02.jpg → FOUND ✅
supported-recyclable__Aluminium_cans__01.jpg → FOUND ✅
supported-hazardous__Button_cells__00.jpg → FOUND ✅
supported-hazardous__Button_cells__03.jpg → FOUND ✅
supported-hazardous__Fluorescent_lamps__01.jpg → FOUND ✅
supported-hazardous__Fluorescent_lamps__02.jpg → FOUND ✅
ambiguous__Pizza_boxes__00.jpg → FOUND ✅
ood-objects__Toy_robots__02.png → FOUND ✅
ood-clothes__Shirts__07.jpg → FOUND ✅
multi-object__Beach_litter__03.jpg → FOUND ✅


In [13]:
from pathlib import Path
import subprocess
import sys

REPO = Path("/content/WasteLens")
BK = "/content/drive/MyDrive/WasteLens/iteration12"

result = subprocess.run(
    [
        sys.executable,
        "src/materialize_manifest.py",
        "--manifest",
        "docs/rejection_experiment/fresh_set_manifest.json",
        "--out",
        "scratch/fresh_set",
        "--drive",
        BK + "/datasets/fresh_set",
        "--report",
        BK + "/datasets/fresh_lock_report.json",
    ],
    cwd=REPO,
    text=True,
    capture_output=True,
)

print(result.stdout)
print(result.stderr)
print("RETURN CODE:", result.returncode)

materialize fresh_set_manifest.json: 131/131 verified, 0 missing, 0 mismatched; report -> /content/drive/MyDrive/WasteLens/iteration12/datasets/fresh_lock_report.json


RETURN CODE: 0


In [14]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "src/iter12_preflight.py"],
    cwd="/content/WasteLens",
    text=True,
    capture_output=True,
)

print(result.stdout)
print(result.stderr)
print("RETURN CODE:", result.returncode)

if result.returncode != 0:
    raise RuntimeError("❌ Iteration-12 preflight FAILED")

print("✅ ITERATION-12 PREFLIGHT PASSED")

PASS adapt_manifest_md5: got acae0933aca123c835c1e55396af182a want acae0933aca123c835c1e55396af182a
PASS adapt_manifest_entries: got 94 want 94
PASS adapt_dir_file_count: got 94 want 94
PASS adapt_files_byte_locked: got {'verified': 94, 'bad_total': 0} want {'verified': 94}
PASS fresh_manifest_md5: got 99540f56efb95e2c26b2b6cc9414ff5d want 99540f56efb95e2c26b2b6cc9414ff5d
PASS fresh_manifest_entries: got 131 want 131
PASS fresh_dir_file_count: got 131 want 131
PASS fresh_files_byte_locked: got {'verified': 131, 'bad_total': 0} want {'verified': 131}
PASS teacher_md5: got 8735019226412c4d67b0a17269dc38ad want 8735019226412c4d67b0a17269dc38ad
preflight: ALL CHECKS PASSED


RETURN CODE: 0
✅ ITERATION-12 PREFLIGHT PASSED


In [1]:
from pathlib import Path
import hashlib

teacher = Path(
    "/content/WasteLens/models/checkpoints/wastelens_rej_shipped_best.keras"
)

drive_teacher = Path(
    "/content/drive/MyDrive/WasteLens/iteration12/checkpoints/wastelens_rej_shipped_best.keras"
)

def md5(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

print("Working-tree teacher exists:", teacher.exists())
print("Drive teacher exists:", drive_teacher.exists())

if teacher.exists():
    print("Working-tree size:", teacher.stat().st_size)
    print("Working-tree MD5:", md5(teacher))

if drive_teacher.exists():
    print("Drive size:", drive_teacher.stat().st_size)
    print("Drive MD5:", md5(drive_teacher))

EXPECTED = "8735019226412c4d67b0a17269dc38ad"

if teacher.exists():
    print("Teacher MD5 PASS:", md5(teacher).lower() == EXPECTED)

if drive_teacher.exists():
    print("Drive teacher MD5 PASS:", md5(drive_teacher).lower() == EXPECTED)

Working-tree teacher exists: False
Drive teacher exists: False


In [2]:
from pathlib import Path

root = Path("/content/drive/MyDrive/WasteLens")

matches = [
    p for p in root.rglob("wastelens_rej_shipped_best.keras")
    if p.is_file()
]

print("Matches found:", len(matches))

for p in matches:
    print(p)

Matches found: 0


In [4]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [5]:
from pathlib import Path
import hashlib

p = Path(
    "/content/drive/MyDrive/WasteLens/iteration12/checkpoints/"
    "wastelens_rej_shipped_best.keras"
)

print("Exists:", p.exists())

if p.exists():
    print("Size:", p.stat().st_size, "bytes")

    h = hashlib.md5()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)

    md5 = h.hexdigest()
    print("MD5:", md5)
    print("MD5 PASS:", md5 == "8735019226412c4d67b0a17269dc38ad")

Exists: True
Size: 13583639 bytes
MD5: 8735019226412c4d67b0a17269dc38ad
MD5 PASS: True


## S3 - materialize datasets (thumb_url only, byte+sha locked, bounded retries)


In [ ]:
for manifest, out, drive_sub, report in (
        (MANIFESTS[0], 'scratch/adaptation_set',
         'datasets/adaptation_set', 'adaptation_lock_report.json'),
        (MANIFESTS[1], 'scratch/fresh_set',
         'datasets/fresh_set', 'fresh_lock_report.json')):
    run([PY, '-u', 'src/materialize_manifest.py',
         '--manifest', manifest, '--out', out,
         '--drive', BK + '/' + drive_sub,
         '--report', BK + '/datasets/' + report])
print('datasets materialized: 94/94 + 131/131 or this cell stopped')


+ /usr/bin/python3 -u src/materialize_manifest.py --manifest docs/rejection_experiment/adaptation_set_manifest.json --out scratch/adaptation_set --drive /content/drive/MyDrive/WasteLens/iteration12/datasets/adaptation_set --report /content/drive/MyDrive/WasteLens/iteration12/datasets/adaptation_lock_report.json
+ /usr/bin/python3 -u src/materialize_manifest.py --manifest docs/rejection_experiment/fresh_set_manifest.json --out scratch/fresh_set --drive /content/drive/MyDrive/WasteLens/iteration12/datasets/fresh_set --report /content/drive/MyDrive/WasteLens/iteration12/datasets/fresh_lock_report.json


SystemExit: STOP rc=1: /usr/bin/python3 -u src/materialize_manifest.py --manifest docs/rejection_experiment/fresh_set_manifest.json --out scratch/fresh_set --drive /content/drive/MyDrive/WasteLens/iteration12/datasets/fresh_set --report /content/drive/MyDrive/WasteLens/iteration12/datasets/fresh_lock_report.json

In [ ]:
import json

report = "/content/drive/MyDrive/WasteLens/iteration12/datasets/fresh_lock_report.json"

with open(report, "r", encoding="utf-8") as f:
    data = json.load(f)

print(json.dumps(data, indent=2))

{
  "generated_utc": "2026-09-30 18:18:27 UTC",
  "manifest": "docs/rejection_experiment/fresh_set_manifest.json",
  "manifest_entries": 131,
  "expected_counts": {
    "supported-organic": 16,
    "supported-recyclable": 21,
    "supported-hazardous": 15,
    "supported-general": 11,
    "ambiguous": 10,
    "ood-objects": 15,
    "ood-clothes": 8,
    "ood-shoes": 8,
    "ood-scenes": 12,
    "multi-object": 15
  },
  "out_dir": "scratch/fresh_set",
  "drive_dir": "/content/drive/MyDrive/WasteLens/iteration12/datasets/fresh_set",
  "verified": 120,
  "resumed_local": 120,
  "restored_from_drive": 0,
  "downloaded": 0,
  "missing": [],
  "sha_mismatch": [
    {
      "file": "supported-organic__Compost__05.jpg",
      "attempts": 4,
      "got_bytes": 493785,
      "want_bytes": 493837,
      "got_sha256": "2c2345225280ca9129cd120a4758dd47e8e9f013d8900f4dff5b0e18352940e3",
      "want_sha256": "159d2456ce057eb0091da6529524f716cc37c06ca73438717be813972abd36be"
    },
    {
      "file"

In [ ]:
import subprocess

cmd = [
    "python", "-u",
    "src/materialize_manifest.py",
    "--manifest", "docs/rejection_experiment/adaptation_set_manifest.json",
    "--out", "scratch/adaptation_set",
    "--drive", "/content/drive/MyDrive/WasteLens/iteration12/datasets/adaptation_set",
    "--report", "/content/drive/MyDrive/WasteLens/iteration12/datasets/adaptation_lock_report.json",
]

print("Running materializer directly...\n")

result = subprocess.run(
    cmd,
    cwd="/content/WasteLens",
    text=True,
    capture_output=True
)

print("===== STDOUT =====")
print(result.stdout)

print("===== STDERR =====")
print(result.stderr)

print("===== RETURN CODE =====")
print(result.returncode)

Running materializer directly...

===== STDOUT =====
materialize adaptation_set_manifest.json: 89/94 verified, 0 missing, 5 mismatched; report -> /content/drive/MyDrive/WasteLens/iteration12/datasets/adaptation_lock_report.json

===== STDERR =====
INCOMPLETE: 0 missing, 5 mismatched - see /content/drive/MyDrive/WasteLens/iteration12/datasets/adaptation_lock_report.json

===== RETURN CODE =====
1


In [ ]:
import json

report = "/content/drive/MyDrive/WasteLens/iteration12/datasets/adaptation_lock_report.json"

with open(report, "r", encoding="utf-8") as f:
    data = json.load(f)

print(json.dumps(data, indent=2))

{
  "generated_utc": "2026-09-30 18:01:27 UTC",
  "manifest": "docs/rejection_experiment/adaptation_set_manifest.json",
  "manifest_entries": 94,
  "expected_counts": {
    "adapt-supported": 60,
    "adapt-ood": 28,
    "adapt-multi": 6
  },
  "out_dir": "scratch/adaptation_set",
  "drive_dir": "/content/drive/MyDrive/WasteLens/iteration12/datasets/adaptation_set",
  "verified": 89,
  "resumed_local": 89,
  "restored_from_drive": 0,
  "downloaded": 0,
  "missing": [],
  "sha_mismatch": [
    {
      "file": "adapt-supported__Button_cells__04.png",
      "attempts": 4,
      "got_bytes": 1446695,
      "want_bytes": 1446695,
      "got_sha256": "b11d136b87bed575e2081b2631082b9c0ee6df0c9b6ea84b93813e164d9e081a",
      "want_sha256": "efcec50933f1144aa79b2a12cd9618373d22cbffdb56a3fb1258bf6f3dc9e757"
    },
    {
      "file": "adapt-supported__Button_cells__05.png",
      "attempts": 4,
      "got_bytes": 725199,
      "want_bytes": 725199,
      "got_sha256": "12794808be07725f6b83e6460a

In [ ]:
from pathlib import Path

drive_dir = Path(
    "/content/drive/MyDrive/WasteLens/iteration12/datasets/adaptation_set"
)

files = [
    "adapt-supported__Button_cells__04.png",
    "adapt-supported__Button_cells__05.png",
    "adapt-supported__Litter__07.jpg",
    "adapt-supported__Paper_towels__06.jpg",
    "adapt-ood__Shoes__03.jpg",
]

for name in files:
    p = drive_dir / name
    print(name, "→", "FOUND ✅" if p.exists() else "MISSING ❌")

adapt-supported__Button_cells__04.png → FOUND ✅
adapt-supported__Button_cells__05.png → FOUND ✅
adapt-supported__Litter__07.jpg → FOUND ✅
adapt-supported__Paper_towels__06.jpg → FOUND ✅
adapt-ood__Shoes__03.jpg → FOUND ✅


## S4 - strict preflight (manifest locks + 94/94 + 131/131 + teacher md5)


In [ ]:
run([PY, '-u', 'src/iter12_preflight.py',
     '--report', BK + '/datasets/head12_preflight.json'])


In [16]:
import subprocess
import sys
from pathlib import Path

REPO = Path("/content/WasteLens")
PY = sys.executable

def run(cmd, log=None):
    cmd = [str(x) for x in cmd]

    print("+", " ".join(cmd))

    if log:
        log_path = Path(log)
        log_path.parent.mkdir(parents=True, exist_ok=True)

        with open(log_path, "w", encoding="utf-8") as f:
            process = subprocess.Popen(
                cmd,
                cwd=REPO,
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True,
                bufsize=1,
            )

            for line in process.stdout:
                print(line, end="")
                f.write(line)

            rc = process.wait()
    else:
        result = subprocess.run(
            cmd,
            cwd=REPO,
            text=True,
        )
        rc = result.returncode

    print("return code:", rc)

    if rc != 0:
        raise RuntimeError(
            f"Command failed with return code {rc}: {' '.join(cmd)}"
        )

    return rc

print("✅ runner ready")
print("PY:", PY)
print("REPO:", REPO)

✅ runner ready
PY: /usr/bin/python3
REPO: /content/WasteLens


In [18]:
import tensorflow as tf
import keras
import sys

print("Python:", sys.version)
print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)
print("tf.keras Keras:", tf.keras.__version__ if hasattr(tf.keras, "__version__") else "no version attr")

Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
TensorFlow: 2.20.0
Keras: 3.13.2
tf.keras Keras: 3.13.2


In [21]:
%pip install -q --upgrade "keras==3.15.1"

^C


In [22]:
import tensorflow as tf
import keras

print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

TensorFlow: 2.20.0
Keras: 3.13.2
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [25]:
from pathlib import Path
import subprocess

REPO = Path("/content/WasteLens")
PINNED = "cf2787b0746506256771a54c3c69feaf6ce3618e"

subprocess.run([
    "git", "clone",
    "https://github.com/Anik00223/WasteLens.git",
    str(REPO)
], check=True)

subprocess.run(
    ["git", "checkout", "--force", PINNED],
    cwd=REPO,
    check=True
)

print(
    subprocess.run(
        ["git", "rev-parse", "HEAD"],
        cwd=REPO,
        capture_output=True,
        text=True,
        check=True,
    ).stdout.strip()
)

CalledProcessError: Command '['git', 'clone', 'https://github.com/Anik00223/WasteLens.git', '/content/WasteLens']' returned non-zero exit status 128.

In [24]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [26]:
from pathlib import Path
import subprocess

REPO = Path("/content/WasteLens")
PINNED = "cf2787b0746506256771a54c3c69feaf6ce3618e"
REPO_URL = "https://github.com/Anik00223/WasteLens.git"

if not REPO.exists():
    print("Cloning WasteLens...")
    subprocess.run(
        ["git", "clone", REPO_URL, str(REPO)],
        check=True
    )
else:
    print("WasteLens already exists.")

subprocess.run(
    ["git", "fetch", "origin", "main"],
    cwd=REPO,
    check=True
)

subprocess.run(
    ["git", "checkout", "--force", PINNED],
    cwd=REPO,
    check=True
)

head = subprocess.run(
    ["git", "rev-parse", "HEAD"],
    cwd=REPO,
    capture_output=True,
    text=True,
    check=True
).stdout.strip()

origin = subprocess.run(
    ["git", "rev-parse", "origin/main"],
    cwd=REPO,
    capture_output=True,
    text=True,
    check=True
).stdout.strip()

print("HEAD       :", head)
print("PINNED     :", PINNED)
print("origin/main:", origin)

assert head == PINNED, "STOP: HEAD != pinned commit"

print("✅ Repository restored and pinned")

WasteLens already exists.
HEAD       : cf2787b0746506256771a54c3c69feaf6ce3618e
PINNED     : cf2787b0746506256771a54c3c69feaf6ce3618e
origin/main: 7e7cb1f84f89c1b45cfe5dde3a796cdd1400ec6d
✅ Repository restored and pinned


In [27]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [28]:
import tensorflow as tf
import keras

print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

TensorFlow: 2.20.0
Keras: 3.13.2
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [29]:
from pathlib import Path
import subprocess
import sys

REPO = Path("/content/WasteLens")
BK = "/content/drive/MyDrive/WasteLens/iteration12"

result = subprocess.run(
    [
        sys.executable,
        "src/restore_teacher_ckpt.py",
        "--drive",
        BK,
    ],
    cwd=REPO,
    text=True,
    capture_output=True,
)

print(result.stdout)
print(result.stderr)

if result.returncode != 0:
    raise RuntimeError("Teacher restore failed.")

print("✅ Teacher restored")

teacher target: models/checkpoints/wastelens_rej_shipped_best.keras
  local md5   : 8735019226412c4d67b0a17269dc38ad  (want 8735019226412c4d67b0a17269dc38ad)
  drive md5   : 8735019226412c4d67b0a17269dc38ad  (/content/drive/MyDrive/WasteLens/iteration12/checkpoints/wastelens_rej_shipped_best.keras)
teacher: working tree already holds the registered checkpoint


✅ Teacher restored


In [30]:
import tensorflow as tf

path = "/content/WasteLens/models/checkpoints/wastelens_rej_shipped_best.keras"

model = tf.keras.models.load_model(path)

print("✅ Teacher loaded successfully")
print("Number of outputs:", len(model.outputs))
print("Output shapes:", [x.shape for x in model.outputs])

TypeError: <class 'keras.src.models.functional.Functional'> could not be deserialized properly. Please ensure that components that are Python object instances (layers, models, etc.) returned by `get_config()` are explicitly deserialized in the model's `from_config()` method.

config={'module': 'keras.src.models.functional', 'class_name': 'Functional', 'config': {}, 'registered_name': 'Functional', 'build_config': {'input_shape': None}, 'compile_config': {'optimizer': {'module': 'keras.optimizers', 'class_name': 'Adam', 'config': {'name': 'adam', 'learning_rate': 0.0010000000474974513, 'weight_decay': None, 'clipnorm': None, 'global_clipnorm': None, 'clipvalue': None, 'use_ema': False, 'ema_momentum': 0.99, 'ema_overwrite_frequency': None, 'loss_scale_factor': None, 'gradient_accumulation_steps': None, 'beta_1': 0.9, 'beta_2': 0.999, 'epsilon': 1e-07, 'amsgrad': False}, 'registered_name': None}, 'loss': {'bins': 'sparse_categorical_crossentropy', 'reject': 'binary_crossentropy'}, 'loss_

In [31]:
import tensorflow as tf
import keras

print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

TensorFlow: 2.20.0
Keras: 3.13.2
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [32]:
import tensorflow as tf

path = "/content/WasteLens/models/checkpoints/wastelens_rej_shipped_best.keras"

try:
    model = tf.keras.models.load_model(path)
    print("✅ TEACHER LOAD SUCCESS")
    print("Outputs:", [x.shape for x in model.outputs])
except Exception as e:
    print("❌ TEACHER LOAD FAILED")
    print(type(e).__name__, e)

❌ TEACHER LOAD FAILED
TypeError <class 'keras.src.models.functional.Functional'> could not be deserialized properly. Please ensure that components that are Python object instances (layers, models, etc.) returned by `get_config()` are explicitly deserialized in the model's `from_config()` method.

config={'module': 'keras.src.models.functional', 'class_name': 'Functional', 'config': {}, 'registered_name': 'Functional', 'build_config': {'input_shape': None}, 'compile_config': {'optimizer': {'module': 'keras.optimizers', 'class_name': 'Adam', 'config': {'name': 'adam', 'learning_rate': 0.0010000000474974513, 'weight_decay': None, 'clipnorm': None, 'global_clipnorm': None, 'clipvalue': None, 'use_ema': False, 'ema_momentum': 0.99, 'ema_overwrite_frequency': None, 'loss_scale_factor': None, 'gradient_accumulation_steps': None, 'beta_1': 0.9, 'beta_2': 0.999, 'epsilon': 1e-07, 'amsgrad': False}, 'registered_name': None}, 'loss': {'bins': 'sparse_categorical_crossentropy', 'reject': 'binary_c

In [33]:
%pip install -q --upgrade "keras==3.15.1"

In [1]:
import tensorflow as tf
import keras

print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

TensorFlow: 2.20.0
Keras: 3.15.1
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
import tensorflow as tf

path = "/content/WasteLens/models/checkpoints/wastelens_rej_shipped_best.keras"

model = tf.keras.models.load_model(path, compile=False)

print("✅ TEACHER LOAD SUCCESS")
print("Outputs:", [x.shape for x in model.outputs])

✅ TEACHER LOAD SUCCESS
Outputs: [(None, 4), (None, 1)]


In [3]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "src/iter12_preflight.py"],
    cwd="/content/WasteLens",
    text=True,
    capture_output=True,
)

print(result.stdout)
print(result.stderr)
print("RETURN CODE:", result.returncode)

PASS adapt_manifest_md5: got acae0933aca123c835c1e55396af182a want acae0933aca123c835c1e55396af182a
PASS adapt_manifest_entries: got 94 want 94
PASS adapt_dir_file_count: got 94 want 94
PASS adapt_files_byte_locked: got {'verified': 94, 'bad_total': 0} want {'verified': 94}
PASS fresh_manifest_md5: got 99540f56efb95e2c26b2b6cc9414ff5d want 99540f56efb95e2c26b2b6cc9414ff5d
PASS fresh_manifest_entries: got 131 want 131
PASS fresh_dir_file_count: got 131 want 131
PASS fresh_files_byte_locked: got {'verified': 131, 'bad_total': 0} want {'verified': 131}
PASS teacher_md5: got 8735019226412c4d67b0a17269dc38ad want 8735019226412c4d67b0a17269dc38ad
preflight: ALL CHECKS PASSED


RETURN CODE: 0


In [4]:
import tensorflow as tf
import keras

print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

TensorFlow: 2.20.0
Keras: 3.15.1
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [8]:
import subprocess
import sys
from pathlib import Path

REPO = Path("/content/WasteLens")
PY = sys.executable
BK = "/content/drive/MyDrive/WasteLens/iteration12"

def run(cmd, log=None):
    cmd = [str(x) for x in cmd]
    print("+", " ".join(cmd))

    if log:
        log_path = Path(log)
        log_path.parent.mkdir(parents=True, exist_ok=True)

        with open(log_path, "w", encoding="utf-8") as f:
            p = subprocess.Popen(
                cmd,
                cwd=REPO,
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True,
                bufsize=1,
            )

            for line in p.stdout:
                print(line, end="")
                f.write(line)

            rc = p.wait()
    else:
        p = subprocess.run(
            cmd,
            cwd=REPO,
            text=True,
        )
        rc = p.returncode

    print("return code:", rc)

    if rc != 0:
        raise RuntimeError(
            f"Command failed with return code {rc}: {' '.join(cmd)}"
        )

    return rc

print("✅ runner ready")
print("REPO:", REPO)
print("PY:", PY)

✅ runner ready
REPO: /content/WasteLens
PY: /usr/bin/python3


## S5 - train seed 42 (6 epochs, FINAL epoch = candidate) + back up immediately


In [10]:
run([PY, '-u', 'src/train_distill_adaptation.py', '--seed', '42'],
    log=BK + '/logs/head12s42_train.log')
run([PY, 'src/backup_iter12.py', 'save', '--drive', BK, '--seed', '42'])
import glob
print('seed-42 checkpoints:', sorted(glob.glob(
    'models/checkpoints/wastelens_rej_head12s42*')))
print('seed-42 evidence docs:', sorted(glob.glob(
    'docs/rejection_experiment/head12s42*')))


+ /usr/bin/python3 -u src/train_distill_adaptation.py --seed 42
[head12] seed=42 epochs=6 lambda=0.5 T=2.0
      dataset locks OK: {"adapt_manifest_md5": "acae0933aca123c835c1e55396af182a", "adapt_dir_file_count": 94, "fresh_manifest_md5": "99540f56efb95e2c26b2b6cc9414ff5d"}
[1/6] Downloading dataset via kagglehub: mostafaabla/garbage-classification
Using Colab cache for faster access to the 'garbage-classification' dataset.
      cached at: /kaggle/input/garbage-classification
[2/6] Scanning image files (dropping classes: clothes, shoes)
      per-class scan:
        battery        945  (expected   945)  ok
        biological     985  (expected   985)  ok
        brown-glass    607  (expected   607)  ok
        cardboard      891  (expected   891)  ok
        green-glass    629  (expected   629)  ok
        metal          769  (expected   769)  ok
        paper         1050  (expected  1050)  ok
        plastic        865  (expected   865)  ok
        trash          697  (expected   6

KeyboardInterrupt: 

## S6 - train seed 43 (identical recipe) + back up immediately


In [ ]:
run([PY, '-u', 'src/train_distill_adaptation.py', '--seed', '43'],
    log=BK + '/logs/head12s43_train.log')
run([PY, 'src/backup_iter12.py', 'save', '--drive', BK, '--seed', '43'])
print('seed-43 checkpoints:', sorted(glob.glob(
    'models/checkpoints/wastelens_rej_head12s43*')))
print('seed-43 evidence docs:', sorted(glob.glob(
    'docs/rejection_experiment/head12s43*')))


## S7 - evaluate both seeds (frozen threshold 0.0702, head12 gate table) + summary


In [ ]:
for seed in ('42', '43'):
    run([PY, '-u', 'src/eval_head_adaptation.py', '--seed', seed,
         '--tag', 'head12', '--protocol',
         'iteration12_distillation_protocol.md',
         '--gates', 'head12_gates.json'],
        log=BK + '/logs/head12s' + seed + '_eval.log')
run([PY, '-u', 'src/eval_head_adaptation.py', '--summary',
     '--tag', 'head12', '--protocol',
     'iteration12_distillation_protocol.md'],
    log=BK + '/logs/head12_summary.log')
run([PY, 'src/backup_iter12.py', 'save', '--drive', BK])
print('eval + decision mirrored; production model/threshold untouched')


## S8 - strict evidence bundle (refuses to build if anything is missing)


In [ ]:
run([PY, 'src/package_head12_evidence.py', '--drive',
     BK + '/evidence', '--out', 'head12_evidence.tar.gz'])
import os
for p in ('head12_evidence.tar.gz',
          BK + '/evidence/head12_evidence.tar.gz'):
    print(p, os.path.getsize(p), 'bytes')


## S9 - SAVE NOW (run any time before the runtime times out)


In [ ]:
run([PY, 'src/backup_iter12.py', 'save', '--drive', BK])
print('full artifact set re-mirrored to', BK)


## S10 - RESTORE from Drive (fresh runtime) + disaster recovery

1. run S0, S1 (pin), S2 (Drive + teacher restore);
2. run this cell - it restores every finished artifact from Drive and re-materializes the datasets (verified files are reused, never re-downloaded);
3. run the S11 status cell - it lists exactly what exists;
4. resume at the FIRST missing step only (S5/S6 for a seed without _final.keras, then S7, S8). Never re-run a completed training cell;
5. nothing on Drive yet? this cell stops with a clear message - start from S3 instead.


In [ ]:
run([PY, 'src/backup_iter12.py', 'restore', '--drive', BK])
for manifest, out, drive_sub, report in (
        (MANIFESTS[0], 'scratch/adaptation_set',
         'datasets/adaptation_set', 'adaptation_lock_report.json'),
        (MANIFESTS[1], 'scratch/fresh_set',
         'datasets/fresh_set', 'fresh_lock_report.json')):
    run([PY, '-u', 'src/materialize_manifest.py',
         '--manifest', manifest, '--out', out,
         '--drive', BK + '/' + drive_sub,
         '--report', BK + '/datasets/' + report])
print('restore complete - run the S11 status cell for the resume point')


## S11 - DISASTER RECOVERY STATUS (pure python)


In [ ]:
import glob, hashlib, os, subprocess

def md5(p):
    h = hashlib.md5()
    with open(p, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

def n_files(d):
    if not os.path.isdir(d):
        return 0
    return len([f for f in os.listdir(d)
                if os.path.isfile(os.path.join(d, f))])

head = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=REPO,
                      capture_output=True, text=True).stdout.strip()
print('HEAD               :', head)
print('pinned as expected :', head == PINNED)
print('git status --short :')
print(subprocess.run(['git', 'status', '--short'], cwd=REPO,
                     capture_output=True, text=True).stdout.strip()
      or '  (clean)')
print('adaptation_set     :', n_files('scratch/adaptation_set'), '/ 94')
print('fresh_set          :', n_files('scratch/fresh_set'), '/ 131')
for rel in MANIFESTS:
    print('manifest md5       :', md5(rel), '<-', rel)
teacher = 'models/checkpoints/wastelens_rej_shipped_best.keras'
print('teacher ckpt md5   :',
      md5(teacher) if os.path.exists(teacher) else 'MISSING',
      '(want 8735019226412c4d67b0a17269dc38ad)')
print('threshold          : 0.0702 (frozen; production untouched)')
print('seed-42 ckpts      :', sorted(os.path.basename(p) for p in
      glob.glob('models/checkpoints/wastelens_rej_head12s42*')))
print('seed-43 ckpts      :', sorted(os.path.basename(p) for p in
      glob.glob('models/checkpoints/wastelens_rej_head12s43*')))
print('seed-42 docs       :', sorted(os.path.basename(p) for p in
      glob.glob('docs/rejection_experiment/head12s42*')))
print('seed-43 docs       :', sorted(os.path.basename(p) for p in
      glob.glob('docs/rejection_experiment/head12s43*')))
print('decision/evidence  :', [p for p in (
    'docs/rejection_experiment/iteration12_decision.md',
    'docs/rejection_experiment/head12_seed_consistency.json',
    'head12_evidence.tar.gz',
    BK + '/evidence/head12_evidence.tar.gz') if os.path.exists(p)])
print('drive root listing :',
      sorted(os.listdir(BK)) if os.path.isdir(BK) else 'MISSING')
print()
print('RESUME RULE: start at S5/S6 for any seed without a _final.keras, then S7 and S8; never re-run a completed training cell.')


## Locks and safety rails (quick reference)

* commit pin: this notebook runs `git checkout --force <PINNED>` and asserts HEAD == PINNED; PINNED is the commit that carries the fixed pipeline.
* dataset manifests: md5 `acae0933aca123c835c1e55396af182a` (94 files) and `99540f56efb95e2c26b2b6cc9414ff5d` (131 images); CRLF checkout pinned by `.gitattributes` so Linux reproduces the registered bytes; the trainer re-checks the lock itself.
* teacher: md5 `8735019226412c4d67b0a17269dc38ad`, restored from Drive and verified before S5/S6 may run.
* threshold: 0.0702 (frozen); the production model is untouched by every cell here.
* every heavy cell stops the run on a non-zero exit; nothing is silently skipped.
